# Análise de Resultados e Ablation Study — Despesas Liquidadas

Análise completa dos resultados do experimento principal e dois ablation studies:

| Ablation | Pergunta respondida |
|---|---|
| **Número de lags** (3 / 6 / 12) | Mais lags sempre melhora? Qual o ponto ótimo por modelo? |
| **Tamanho do holdout** (3 / 6 / 12 meses) | O ranking dos modelos muda com períodos de avaliação maiores? |

**Pré-requisito**: execute primeiro `python experiment.py` (ou o notebook 01).

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
import pandas as pd
import numpy as np
from IPython.display import display, HTML, Image
warnings.filterwarnings('ignore')

from config import RESULTS_DIR, PLOTS_DIR
from reports.plots import (
    plotar_resumo_modelos, plotar_vencedores,
    plotar_distribuicao_rmse,
    plotar_ablation_lags, plotar_ablation_holdout,
)

print('Setup OK')

## 1. Carregar Resultados do Experimento

In [ ]:
def carregar_mais_recente(prefixo: str) -> pd.DataFrame:
    arquivos = sorted(RESULTS_DIR.glob(f"{prefixo}_*.csv"))
    if not arquivos:
        raise FileNotFoundError(
            f"Nenhum arquivo {prefixo}_*.csv encontrado. "
            "Execute experiment.py primeiro."
        )
    arq = arquivos[-1]
    print(f"Carregando: {arq.name}")
    return pd.read_csv(arq)

metricas_df = carregar_mais_recente('metricas')
params_df   = carregar_mais_recente('parametros')
prev_df     = carregar_mais_recente('previsoes')

print(f"\nSéries processadas : {prev_df['Codigo'].nunique()}")
modelos_validos = [m for m in metricas_df['Modelo'].unique() if isinstance(m, str)]
print(f"Modelos avaliados  : {sorted(modelos_validos)}")

## 2. Visão Geral dos Resultados

In [ ]:
# Estatísticas descritivas por modelo
resumo = (
    metricas_df[metricas_df['Modelo'] != 'N/A']
    .groupby('Modelo')[['RMSE','MAE','MAPE']]
    .agg(['mean','median','std','min','max'])
    .round(2)
)
display(resumo)

In [ ]:
# Ranking de vitórias
vitorias = prev_df['Melhor_Modelo'].value_counts().reset_index()
vitorias.columns = ['Modelo', 'Vitórias']
vitorias['%'] = (vitorias['Vitórias'] / len(prev_df) * 100).round(1)
display(vitorias.style.hide(axis='index'))

In [ ]:
p = plotar_resumo_modelos(metricas_df[metricas_df['Modelo'] != 'N/A'])
display(Image(str(p)))

In [ ]:
p = plotar_distribuicao_rmse(metricas_df[metricas_df['Modelo'] != 'N/A'])
display(Image(str(p)))

In [ ]:
p = plotar_vencedores(prev_df)
display(Image(str(p)))

## 3. Previsões Finais — Jul/Ago 2025

In [ ]:
cols = [
    'Codigo', 'Nome', 'Grupo_Sinal', 'Melhor_Modelo',
    'RMSE_Teste', 'MAPE_Teste_Pct',
    'Ultimo_Real_fmt', 'Prev_Jul_2025_fmt', 'Prev_Ago_2025_fmt',
    'Var_Jul_vs_Jun_Pct', 'Var_Ago_vs_Jul_Pct',
]

def cor_var(val):
    if pd.isna(val): return ''
    return 'color: green' if val >= 0 else 'color: red'

display(
    prev_df[cols]
    .style
    .format({
        'RMSE_Teste'        : '{:,.2f}',
        'MAPE_Teste_Pct'    : '{:.2f}%',
        'Var_Jul_vs_Jun_Pct': '{:+.2f}%',
        'Var_Ago_vs_Jul_Pct': '{:+.2f}%',
    })
    .applymap(cor_var, subset=['Var_Jul_vs_Jun_Pct','Var_Ago_vs_Jul_Pct'])
    .hide(axis='index')
)

## 4. Gráficos por Série

In [ ]:
plots = sorted(PLOTS_DIR.glob('serie_*.png'))
print(f'{len(plots)} gráficos encontrados em {PLOTS_DIR.name}/')

for plot in plots:
    display(HTML(f'<h4 style="color:#1B3A5C">{plot.stem.replace("serie_","Ação: ")}</h4>'))
    display(Image(str(plot), width=900))

---
## 5. Ablation Study

> ⚠️ O ablation roda os modelos múltiplas vezes. **Pode demorar vários minutos.**

### 5a. Executar Ablation

In [ ]:
from ablation import rodar_ablation_completo

abl = rodar_ablation_completo(verbose=True)
lags_df    = abl['lags_df']
holdout_df = abl['holdout_df']

### 5b. Carregar Ablation já executado (alternativa)

In [ ]:
# Remova os '#' abaixo para carregar CSVs de uma execução anterior.
# lags_df    = carregar_mais_recente('ablation_lags')
# holdout_df = carregar_mais_recente('ablation_holdout')
print("Célula desativada — remova os '#' para usar CSVs existentes.")

### 5c. Ablation — Número de Lags

In [ ]:
tab_lags = (
    lags_df.groupby(['N_Lags','Modelo'])['RMSE']
    .agg(Media='mean', Mediana='median', DesvPad='std')
    .round(2).reset_index()
)
display(tab_lags.style.hide(axis='index'))

In [ ]:
p = plotar_ablation_lags(lags_df)
display(Image(str(p)))

In [ ]:
melhor_lags = (
    lags_df.groupby(['Modelo','N_Lags'])['RMSE'].mean()
    .reset_index()
    .sort_values(['Modelo','RMSE'])
    .groupby('Modelo').first()
    .reset_index()[['Modelo','N_Lags','RMSE']]
    .rename(columns={'N_Lags':'Melhor_N_Lags','RMSE':'RMSE_Médio'})
)
print('Configuração de lags com menor RMSE médio por modelo:')
display(melhor_lags.style.hide(axis='index'))

### 5d. Ablation — Tamanho do Holdout

In [ ]:
tab_hout = (
    holdout_df.groupby(['N_Holdout','Modelo'])['RMSE']
    .agg(Media='mean', Mediana='median', DesvPad='std')
    .round(2).reset_index()
)
display(tab_hout.style.hide(axis='index'))

In [ ]:
p = plotar_ablation_holdout(holdout_df)
display(Image(str(p)))

In [ ]:
for h, g in (
    holdout_df.groupby(['N_Holdout','Modelo'])['RMSE']
    .mean().reset_index()
    .sort_values(['N_Holdout','RMSE'])
    .groupby('N_Holdout')
):
    print(f"\nHoldout = {h} meses:")
    for pos, row in enumerate(g.itertuples(), 1):
        print(f"  {pos}º {row.Modelo:6s}  RMSE médio = {row.RMSE:,.0f}")

---
## 6. Síntese

In [ ]:
print('=' * 58)
print('  SÍNTESE DO EXPERIMENTO E ABLATION STUDY')
print('=' * 58)

met_validos = metricas_df[metricas_df['Modelo'] != 'N/A']
melhor = met_validos.groupby('Modelo')['RMSE'].mean().idxmin()
rmse_m = met_validos.groupby('Modelo')['RMSE'].mean().min()
print(f"\n► Melhor modelo (RMSE médio): {melhor} ({rmse_m:,.0f})")

if not lags_df.empty:
    ml = lags_df[lags_df['Modelo'] != 'ARIMA'].groupby('N_Lags')['RMSE'].mean().idxmin()
    print(f"► Melhor número de lags (LR/SVR/MLP): {ml}")

if not holdout_df.empty:
    rankings_por_holdout = {}
    for h, g in holdout_df.groupby('N_Holdout'):
        rankings_por_holdout[h] = g.groupby('Modelo')['RMSE'].mean().idxmin()
    primeiros = list(rankings_por_holdout.values())
    estavel   = len(set(primeiros)) == 1
    print(f"► Ranking estável com diferentes holdouts: {'Sim' if estavel else 'Não — há inversões'}")
    for h, mod in rankings_por_holdout.items():
        print(f"   holdout={h}m → 1º lugar: {mod}")

print('=' * 58)